In [149]:
import os
import polars as pl
from tqdm.notebook import tqdm

pl.Config(tbl_rows=50)

In [150]:
data_path = '../data/meds_outliers/'

In [151]:
data = pl.read_parquet('../data/meds_outliers/data/train/1.parquet')

In [152]:
# pl.Config(fmt_str_lengths=100000)
# data.with_columns(pl.col('code').str.split('//').list.len().alias('length')).group_by(['code_type','length']).first()['code_type','length','code']

In [153]:
data.filter(pl.col('code_type') == 'ICU-FLUID-OUTPUT').head(1)

subject_id,seq_id,out_id,er_id,hadm_id,icustay_id,disch_id,time,code,numeric_value,text_value,itemid,died_in_hosp,icu_los,admission_type,admission_location,discharge_location,diag_version,diag_icd_code,diag_seq_num,drg_severity,drg_mortality,drg_type,drg_code,priority,specimen_id,lab_lower_limit,lab_upper_limit,lab_flag,lab_unit,lab_itemid,gender,route,frequency,doses_per_24_hrs,medication,proc_seq_num,proc_version,proc_icd_code,micro_specimen_id,micro_org_name,micro_test_name,micro_spec_type_desc,micro_test_itemid,icu_care_unit,category,label,abbreviation,rate,unit,amount,amountuom,ordercategorydescription,ordercategoryname,secondaryordercategoryname,ordercomponenttypedescription,table,race,code_type,icd9_to_icd10_d,icd9_to_icd10_p,clean_medication,lab_label,lab_fluid,lab_category,lab_description,lab_frequency,time_diff,numeric_value/is_inlier
i64,f64,f64,f64,f64,f64,f64,datetime[μs],str,f64,str,f64,f64,f64,str,str,str,f64,str,f64,f64,f64,str,f64,str,f64,f64,f64,str,str,f64,str,str,str,f64,str,f64,f64,str,f64,str,str,str,f64,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,bool
10052769,2.2087051e7,null,null,2.2087051e7,3.883265e7,null,2124-04-26 13:41:00,"""ICU-FLUID-OUTPUT//226560//Void""",400.0,null,226560.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Output""","""Void""","""Void""",null,null,null,null,null,null,null,null,"""icu/inputevents""",null,"""ICU-FLUID-OUTPUT""",null,null,"""UNK""",null,null,null,null,null,0.007639,true


In [154]:
descemb_mapping = {'MICROBIOLOGY': ['micro_test_name'],
'DIAGNOSIS-ICD': ['code_type','icd9_to_icd10_d'],
'MEDS_DEATH': ['code_type'],
'EMERGENCY-END': ['code_type'],
'EMERGENCY-START': ['code_type'],
'ADMISSION-AT-HOSPITAL': ['code_type'],
'MEDICATION': ['clean_medication','route','doses_per_24_hrs'],
'DISCHARGE-FROM-ICU': ['code_type'],
'OUTPATIENT-END': ['code_type'],
'ICU-CHART': ['label','numeric_value' 'unitname'],
'DISCHARGE-FROM-HOSPITAL': ['code_type'],
'ADMISSION-LOCATION': ['code_type'], #  and use .split('//')[-1] as second element of the list]
'LAB': ['lab_label', 'numeric_value', 'lab_unit'],
'RACE': ['code_type','race'],
'AGE_AT_ADMISSION': ['code_type','numeric_value'],
'PROCEDURE-ICD': ['code_type','icd9_to_icd10_p'],
'TIME-GAP': ['code_type','numeric_value'],
'DRG': ['code_type','drg_type','drg_code', 'drg_mortality'],
'ADMISSION-TYPE':['code_type'], #  and use .split('//')[-1] as second element of the list]
'ADMISSION-AT-ICU': ['code_type'],
'ICU-INFUSION': ['label','numeric_value', 'amountuom'],
'DISCHARGE-lOCATION': ['code_type','discharge_location'],
'GENDER': ['code_type','gender'],
'ICU-FLUID-OUTPUT': ['label','numeric_value' 'unitname'],
'OUTPATIENT-START': ['code_type'],
'ICU-PROCEDURE': ['label'],
}

In [155]:
import polars as pl

def dsva_number(x):
    """
    Digit-Split Value Aggregation with max 3 decimals:
    12.34567  -> '1 2 . 3 5'
    5         -> '5'
    5.1       -> '5 . 1'
    """
    if x is None:
        return None
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None

    # round to 3 decimal places
    v = round(v, 3)

    # format to 3 decimals, then strip trailing zeros and dot
    s = f"{v:.3f}".rstrip("0").rstrip(".")

    # DSVA: split into individual characters separated by spaces
    return " ".join(list(s))


def dsva_expr(col: pl.Expr) -> pl.Expr:
    return (
        col.cast(pl.Float64)
           .map_elements(dsva_number, return_dtype=pl.Utf8)
    )


In [156]:
gender_norm = (
    pl.when(pl.col("gender").str.to_lowercase() == "m")
      .then(pl.lit("male"))
    .when(pl.col("gender").str.to_lowercase() == "f")
      .then(pl.lit("female"))
    .otherwise(pl.col("gender"))
)


In [157]:
def normalize_code_type():
    return (
        pl.col("code_type")
        .str.replace_all(r"[_\-]+", " ")   # replace underscores and dashes with space
        .str.to_lowercase()
        .str.strip()
    )

In [158]:
def add_descemb(df: pl.DataFrame) -> pl.DataFrame:
    ct = pl.col("code_type")
    code = pl.col("code")

    # clean code type: remove - and _, lowercase
    code_type_clean = (
        ct.str.replace_all(r"[_\-]+", " ")
          .str.to_lowercase()
          .str.strip_chars()
    )

    # last element after the last "//"
    code_tail = code.str.extract(r"([^/]+)$").str.to_lowercase()

    descemb_expr = (
        pl.when(ct == "MICROBIOLOGY")
          .then(
              pl.col("micro_test_name").str.to_lowercase()
          )

        .when(ct == "DIAGNOSIS-ICD")
          .then(
              pl.concat_str(
                  [pl.lit("diagnosis icd 10 code"), pl.col("icd9_to_icd10_d")],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "MEDS_DEATH")
          .then(code_type_clean)

        .when(ct == "EMERGENCY-END")
          .then(code_type_clean)

        .when(ct == "EMERGENCY-START")
          .then(code_type_clean)

        .when(ct == "ADMISSION-AT-HOSPITAL")
          .then(code_type_clean)

        .when(ct == "MEDICATION")
          .then(
              pl.concat_str(
                  [
                      pl.col("clean_medication"),
                      pl.col("route"),
                      pl.col("doses_per_24_hrs")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-FROM-ICU")
          .then(code_type_clean)

        .when(ct == "OUTPATIENT-END")
          .then(code_type_clean)

        .when(ct == "ICU-CHART")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("unitname")  # change if your col is named differently
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-FROM-HOSPITAL")
          .then(code_type_clean)

        .when(ct == "ADMISSION-LOCATION")
          .then(
              pl.concat_str(
                  [code_type_clean, code_tail],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "LAB")
          .then(
              pl.concat_str(
                  [
                      pl.col("lab_label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("lab_unit")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "RACE")
          .then(
              pl.concat_str(
                  [code_type_clean, pl.col("race")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "AGE_AT_ADMISSION")
          .then(
              pl.concat_str(
                  [
                      code_type_clean,
                      dsva_expr(pl.col("numeric_value"))
                  ],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "PROCEDURE-ICD")
          .then(
              pl.concat_str(
                  [pl.lit("procedure icd 10 code"), pl.col("icd9_to_icd10_p")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )


        .when(ct == "TIME-GAP")
          .then(
              pl.concat_str(
                  [code_type_clean, dsva_expr(pl.col("numeric_value"))],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "DRG")
          .then(
              pl.concat_str(
                  [   code_type_clean,
                      pl.col("drg_type"),
                      pl.col("drg_code"),
                      pl.col("drg_mortality")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "ADMISSION-TYPE")
          .then(
              pl.concat_str(
                  [code_type_clean, code_tail],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "ADMISSION-AT-ICU")
          .then(code_type_clean)

        .when(ct == "ICU-INFUSION")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("amountuom")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-lOCATION")
          .then(
              pl.concat_str(
                  [code_type_clean, pl.col("discharge_location")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "GENDER")
          .then(
              pl.concat_str(
                  [code_type_clean, gender_norm],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "ICU-FLUID-OUTPUT")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("unitname")  # change if needed
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "OUTPATIENT-START")
          .then(code_type_clean)

        .when(ct == "ICU-PROCEDURE")
          .then(
              pl.col("label").str.to_lowercase()
          )

        .otherwise(pl.lit(None))
        .alias("descemb")
    )



    return df.with_columns(descemb_expr)#['subject_id',
#                                         'seq_id',
#                                         'out_id',
#                                         'er_id',
#                                         'hadm_id',
#                                         'icustay_id',
#                                         'disch_id',
#                                         'time',
#                                         'code',
#                                         'code_type',
#                                         'descemb']


In [159]:
genhpf_mapping = {'MICROBIOLOGY': ['code_type','micro_test_name', 'micro_spec_type_desc'],
'DIAGNOSIS-ICD': ['code_type','icd9_to_icd10_d'],
'MEDS_DEATH': ['code_type'],
'EMERGENCY-END': ['code_type'],
'EMERGENCY-START': ['code_type'],
'ADMISSION-AT-HOSPITAL': ['code_type'],
'MEDICATION': ['code_type','clean_medication','route','frequency','doses_per_24_hrs'],
'DISCHARGE-FROM-ICU': ['code_type'],
'OUTPATIENT-END': ['code_type'],
'ICU-CHART': ['code_type','category','label','numeric_value', 'unitname'],
'DISCHARGE-FROM-HOSPITAL': ['code_type'],
'ADMISSION-LOCATION': ['code_type'], #  and use .split('//')[-1] as second element of the list]
'LAB': ['code_type','lab_label', 'priority' ,'numeric_value', 'lab_unit','lab_lower_limit', 'lab_upper_limit', 'lab_flag'],
'RACE': ['code_type','race'],
'AGE_AT_ADMISSION': ['code_type','numeric_value'],
'PROCEDURE-ICD': ['code_type','icd9_to_icd10_p'],
'TIME-GAP': ['code_type','numeric_value'],
'DRG': ['code_type','drg_type','drg_code', 'drg_mortality'],
'ADMISSION-TYPE':['code_type'], #  and use .split('//')[-1] as second element of the list]
'ADMISSION-AT-ICU': ['code_type'],
'ICU-INFUSION': ['code_type', 'category', 'label','numeric_value', 'amountuom'],
'DISCHARGE-lOCATION': ['code_type','discharge_location'],
'GENDER': ['code_type','gender'],
'ICU-FLUID-OUTPUT': ['code_type','category','label','numeric_value', 'unitname'],
'OUTPATIENT-START': ['code_type'],
'ICU-PROCEDURE': ['code_type','label'],
}

In [160]:
FEATURE_NAME_MAP = {
    "micro_test_name": "test name",
    "micro_spec_type_desc": "test description",
    "clean_medication": "medication name",
    "doses_per_24_hrs": "dose per 24 hours",
    "numeric_value": "value",
    "lab_label": "lab name",
    "lab_unit": "unit",
    "lab_flag": "flag",
    "amountuom": "unit",
    "unitname": "unit",
    "icd9_to_icd10_d": "value",
    "icd9_to_icd10_p": "value",
}

In [161]:
CODETYPE = (
    pl.col("code_type")
      .str.replace_all(r"[_\-]+", " ")
      .str.to_lowercase()
      .str.strip_chars()
)

In [162]:
def value_expr(col: pl.Expr, name: str) -> pl.Expr:
    if name in ["numeric_value"]:
        return dsva_expr(col)
    if name in ["lab_lower_limit", "lab_upper_limit"]:
        return dsva_expr(col)
    if name in ["icd9_to_icd10_d", "icd9_to_icd10_p"]:
        return col  # already textual ICD10 code
    return col  # textual columns

In [163]:
def feature_pair(feature_col: str) -> pl.Expr:
    """Return expression: '<feature name> <value>' """
    feat_name = FEATURE_NAME_MAP.get(feature_col, feature_col.replace("_", " ").replace("-", " "))
    feat_name = feat_name.lower()

    col_expr = pl.col(feature_col)
    col_expr = value_expr(col_expr, feature_col)

    return pl.concat_str([pl.lit(f"{feat_name}: "), col_expr], separator="", ignore_nulls=True)


In [164]:
def build_genhpf_desc(df: pl.DataFrame, mapping: dict) -> pl.DataFrame:
    ct_clean = CODETYPE
    code_tail = pl.col("code").str.extract(r"([^/]+)$").str.to_lowercase()

    # Build master expression
    expr = pl.lit("")  # will be overridden per code_type

    for code_type, cols in mapping.items():
        parts = []

        # always start with cleaned code type:
        parts.append(
                    pl.concat_str(
                        [pl.lit("event type: "), ct_clean],
                        separator="",
                        ignore_nulls=True
                    )
                )

        # add each feature pair
        for col in cols:
            if col == "code_type":
                continue
            parts.append(feature_pair(col))

        # build text for this code_type
        text_expr = (
            pl.concat_str(parts, separator=", ", ignore_nulls=True)
              .str.to_lowercase()
        )

        # add conditional branch
        expr = (
            pl.when(pl.col("code_type") == code_type)
              .then(text_expr)
              .otherwise(expr)
        )

    return df.with_columns(expr.alias("genhpf"))['subject_id',
                                        'seq_id',
                                        'out_id',
                                        'er_id',
                                        'hadm_id',
                                        'icustay_id',
                                        'disch_id',
                                        'code',
                                        'time',
                                        'descemb',
                                        'genhpf']


In [165]:
# data_path = '../data/meds_outliers/data/train/'
# out_path = '../data/descemb_genhpf/data/train/'
# d_item = pl.read_csv('../resources/mimic-mapping/d_items.csv',
#                      infer_schema_length=100000,
#                     columns=['itemid','unitname'])

# for shard in tqdm(os.listdir(data_path)):
#     data = pl.read_parquet(os.path.join(data_path,shard))
#     data = data.with_columns(pl.col("itemid").cast(pl.Int64))
#     data = data.join(d_item, on='itemid', how='left')
#     data = add_descemb(data)
#     data = build_genhpf_desc(data, genhpf_mapping)
#     data.write_parquet(os.path.join(out_path,shard))

In [166]:
limits = {
    'within24_query': {512:  ['w24_start_512',  'w24_end_512' ]},
    'within48_query': {512:  ['w48_start_512',  'w48_end_512' ]},
    'within_stay_query': {512:  ['wStay_start_512',  'wStay_end_512' ]},

    }

In [167]:
# from datasets import Dataset, Features, Sequence, Value
# data_idx = pl.read_parquet('../downstream_idx.parquet')

# def gen():
#     for i in range(len(data_idx)):
#         subject_id = data_idx[i]['subject_id'][0]
#         icustay_id = data_idx[i]['icustay_id'][0]
#         shard = data_idx[i]['shard'][0]
        
#         w24_start = data_idx[i]['w24_start_512'][0]
#         w24_end = data_idx[i]['w24_end_512'][0]
        
#         w48_start = data_idx[i]['w48_start_512'][0]
#         w48_end = data_idx[i]['w48_end_512'][0]
        
#         wstay_start = data_idx[i]['wStay_start_512'][0]
#         wstay_end = data_idx[i]['wStay_end_512'][0]
        
#         file = pl.read_parquet(os.path.join('../data/descemb_genhpf/data/train/',shard))
#         seq = file.filter(pl.col('subject_id') == subject_id)
        
        

#         yield {
#             "subject_id": subject_id,
#             "icustay_id": icustay_id,
            
#             "within24_descemb": seq[w24_start:w24_end,:]['descemb'].to_list(),
#             "within48_descemb": seq[w48_start:w48_end,:]['descemb'].to_list(),
#             "within_stay_descemb": seq[wstay_start:wstay_end,:]['descemb'].to_list(),
            
#             "within24_genhpf": seq[w24_start:w24_end,:]['genhpf'].to_list(),
#             "within48_genhpf": seq[w48_start:w48_end,:]['genhpf'].to_list(),
#             "within_stay_genhpf": seq[wstay_start:wstay_end,:]['genhpf'].to_list(),
#             }

# features = Features({
#     "subject_id":     Value("int32"),
#     "icustay_id":     Value("int32"),
#     "within24_descemb":    Sequence(Value("string")),
#     "within48_descemb":    Sequence(Value("string")),
#     "within_stay_descemb":    Sequence(Value("string")),
    
#     "within24_genhpf":    Sequence(Value("string")),
#     "within48_genhpf":    Sequence(Value("string")),
#     "within_stay_genhpf":    Sequence(Value("string"))})

# ds_arrow = Dataset.from_generator(
#     gen,
#     features=features,
#     writer_batch_size=1000  # tune for shard sizes
# )

# # write Arrow shards to disk (memory-mappable)
# ds_arrow.save_to_disk("desc_gen_dataset")

# # optional: set PyTorch formatting
# # ds_arrow.set_format(type="torch")



# # # later / in training script:
# # from datasets import load_from_disk
# # train = load_from_disk("ehr_arrow_dataset")
# # train.set_format(type="torch")

In [168]:
from datasets import load_from_disk

In [169]:
import os
import polars as pl
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from transformers import AutoTokenizer


class DescGenDataset(Dataset):
    def __init__(
        self,
        dataset_path: str,
        data_idx_path: str,
        task: str = "y_mort",
        main_window: str = "within48_descemb",  
        split: str = "train",
        max_word_len: int = 32,                 
        max_events: int = None,                 
    ) -> None:

        self.task = task
        self.main_window = main_window
        self.max_word_len = max_word_len
        self.max_events = max_events
        

        self.data_idx = pl.scan_parquet(data_idx_path).collect()
        self.data_idx = self.data_idx.filter(pl.col("split") == split)
        self.data_idx = self.data_idx.filter(~pl.col("subject_id").is_in([15409850,16816440,18757959]) )


        self.hf_dataset = load_from_disk(dataset_path)

        subj_ids = self.hf_dataset["subject_id"]
        icu_ids = self.hf_dataset["icustay_id"]
        self._hf_index = {
            (int(s), int(i)): idx for idx, (s, i) in enumerate(zip(subj_ids, icu_ids))
        }


        self.tokenizer = AutoTokenizer.from_pretrained(
            "google/bert_uncased_L-2_H-128_A-2"
        )

    def __len__(self) -> int:
        return len(self.data_idx)

    def __getitem__(self, idx: int):
        row = self.data_idx.row(idx, named=True)
        subject_id = int(row["subject_id"])
        icustay_id = int(row["icustay_id"])
        label = row[self.task]

        ex = self.hf_dataset[idx]

        events = ex[self.main_window]  

        if self.max_events is not None and len(events) > self.max_events:
            events = events[: self.max_events]

        enc = self.tokenizer(
            events,
            padding="max_length",
            truncation=True,
            max_length=self.max_word_len,
            add_special_tokens=True,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"]         
#         token_type_ids = enc["token_type_ids"]
        attention_mask = enc["attention_mask"]

        seq_len = torch.tensor(len(events), dtype=torch.long)
        label = torch.tensor(label, dtype=torch.float)  

        return {
            "input_ids": input_ids,              
#             "token_type_ids": token_type_ids,    
            "attention_mask": attention_mask,                   
            "label": label                      
        }

In [170]:
ds = DescGenDataset(data_idx_path='../downstream_idx.parquet',
                    dataset_path='./desc_gen_dataset/',
                    main_window='within48_descemb',
                    split='train',
                    max_word_len=16)

Loading dataset from disk:   0%|          | 0/24 [00:00<?, ?it/s]

In [171]:
import torch

class DescGenCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):

        # remove empty samples
        batch = [b for b in batch if b["input_ids"] is not None]
        if len(batch) == 0:
            return {}

        lengths = [b["input_ids"].shape[0] for b in batch]
        max_S = max(lengths)
        W = batch[0]["input_ids"].shape[1]
        B = len(batch)

        input_ids = torch.full((B, max_S, W), self.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((B, max_S, W), dtype=torch.long)
        seq_len = torch.tensor(lengths, dtype=torch.long)
        labels = torch.stack([b["label"] for b in batch])

        for i, b in enumerate(batch):
            S_i = b["input_ids"].shape[0]
            input_ids[i, :S_i] = b["input_ids"]
            attention_mask[i, :S_i] = b["attention_mask"]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "seq_len": seq_len,
            "label": labels,
        }

collate_fn = DescGenCollator(ds.tokenizer.pad_token_id)

In [172]:
dl = DataLoader(dataset=ds, batch_size=32,collate_fn=collate_fn, shuffle=True)

In [173]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig


class BertEventEncoder(nn.Module):
    def __init__(
        self,
        bert_model_name: str = "google/bert_uncased_L-2_H-128_A-2",
        pred_embed_dim: int = 128,
        init_bert_random: bool = False,
    ):
        super().__init__()

        if init_bert_random:
            config = AutoConfig.from_pretrained(bert_model_name)
            self.bert = AutoModel.from_config(config)
        else:
            self.bert = AutoModel.from_pretrained(bert_model_name)

        hidden_size = self.bert.config.hidden_size
        self.post_encode_proj = nn.Linear(hidden_size, pred_embed_dim)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B, S, W = input_ids.shape

        flat_ids = input_ids.view(B * S, W)
        flat_mask = attention_mask.view(B * S, W)

        outputs = self.bert(
            input_ids=flat_ids,
            attention_mask=flat_mask,
        )
        # CLS embedding per event
        cls_emb = outputs.last_hidden_state[:, 0, :] 

        event_emb = self.post_encode_proj(cls_emb)  
        event_emb = event_emb.view(B, S, -1)        
        return event_emb

In [174]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class GRUEventHead(nn.Module):

    def __init__(self, pred_embed_dim: int, pred_hidden_dim: int, max_event_len: int,
                 n_layers: int = 1, dropout: float = 0.1, task: str = "binary"):
        super().__init__()
        self.pred_embed_dim = pred_embed_dim
        self.pred_hidden_dim = pred_hidden_dim
        self.n_layers = n_layers
        self.max_event_len = max_event_len
        self.task = task

        self.model = nn.GRU(
            input_size=self.pred_embed_dim,
            hidden_size=self.pred_hidden_dim,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=False,
            num_layers=self.n_layers,
        )

        out_dim = 18 if task == "diagnosis" else 1
        self.final_proj = nn.Linear(self.pred_hidden_dim, out_dim)

    def pack_pad_seq(self, x: torch.Tensor, lengths: torch.Tensor):
        lengths = lengths.view(-1).cpu()
        lengths[lengths > self.max_event_len] = self.max_event_len

        packed = pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        output, _ = self.model(packed)
        output_seq, output_len = pad_packed_sequence(
            output, batch_first=True, padding_value=0.0
        )
        return output_seq, output_len

    def forward(self, x: torch.Tensor, seq_len: torch.Tensor) -> torch.Tensor:
       
        self.model.flatten_parameters()

        output_seq, _ = self.pack_pad_seq(x, seq_len) 
        i = range(x.size(0))
        last_hidden = output_seq[i, -1, :]             

        logits = self.final_proj(last_hidden)          
        if logits.shape[-1] == 1:
            logits = logits.squeeze(-1)                
        return logits

In [175]:
import lightning as lt
import torch
import torch.nn as nn
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision


class DescGenEvalModel(lt.LightningModule):
    def __init__(
        self,
        config,
        lr: float = 2e-5,
        wd: float = 0.001,
        max_epochs: int = 100,
        dropout: float = 0.1,
        freeze: bool = False,
    ):
        super().__init__()
        self.save_hyperparameters()

        bert_model_name = getattr(config, "bert_model_name", "google/bert_uncased_L-2_H-128_A-2")
        pred_embed_dim = getattr(config, "pred_embed_dim", 128)
        pred_hidden_dim = getattr(config, "pred_hidden_dim", 128)
        max_event_len = getattr(config, "max_event_len", 511)
        task = getattr(config, "task", "binary")

        self.encoder = BertEventEncoder(
            bert_model_name=bert_model_name,
            pred_embed_dim=pred_embed_dim,
            init_bert_random=getattr(config, "init_bert_random", False),
        )
        self.classifier = GRUEventHead(
            pred_embed_dim=pred_embed_dim,
            pred_hidden_dim=pred_hidden_dim,
            max_event_len=max_event_len,
            n_layers=getattr(config, "rnn_layer", 1),
            dropout=dropout,
            task=task,
        )

        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False
            for p in self.classifier.parameters():
                p.requires_grad = True

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs

        self.criterion = nn.BCEWithLogitsLoss()

        self.train_step_preds = []
        self.train_step_label = []
        self.val_step_preds = []
        self.val_step_label = []
        self.test_step_preds = []
        self.test_step_label = []

        self.train_auroc = BinaryAUROC()
        self.train_auprc = BinaryAveragePrecision()
        self.val_auroc = BinaryAUROC()
        self.val_auprc = BinaryAveragePrecision()
        self.test_auroc = BinaryAUROC()
        self.test_auprc = BinaryAveragePrecision()

    def forward(self, input_ids, attention_mask, seq_len=None, labels=None):
        event_emb = self.encoder(input_ids=input_ids, attention_mask=attention_mask)  

        if seq_len is None:
            event_mask = attention_mask.any(dim=-1)     
            seq_len = event_mask.sum(dim=-1)             
        logits = self.classifier(event_emb, seq_len)       
        return logits

    def training_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)

        pos_score = torch.sigmoid(logits)

        self.train_step_label.append(y)
        self.train_step_preds.append(pos_score)

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_train_epoch_end(self) -> None:
        if len(self.train_step_label) == 0:
            return
        y = torch.cat(self.train_step_label)
        pos_score = torch.cat(self.train_step_preds)

        auroc = self.train_auroc(pos_score, y.long())
        auprc = self.train_auprc(pos_score, y.long())

        self.log("train_auroc", auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log("train_auprc", auprc, on_epoch=True, logger=True, prog_bar=True)

        self.train_step_label.clear()
        self.train_step_preds.clear()

    def validation_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.val_step_label.append(y)
        self.val_step_preds.append(pos_score)

        self.log("val_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_validation_epoch_end(self, *args, **kwargs) -> None:
        if len(self.val_step_label) == 0:
            return
        y = torch.cat(self.val_step_label)
        pos_score = torch.cat(self.val_step_preds)

        auroc = self.val_auroc(pos_score, y.long())
        auprc = self.val_auprc(pos_score, y.long())

        self.log("val_auroc", auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log("val_auprc", auprc, on_epoch=True, logger=True, prog_bar=True)

        self.val_step_label.clear()
        self.val_step_preds.clear()

    def test_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.test_step_label.append(y)
        self.test_step_preds.append(pos_score)

        self.log("test_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_test_epoch_end(self, *args, **kwargs) -> None:
        if len(self.test_step_label) == 0:
            return
        y = torch.cat(self.test_step_label)
        pos_score = torch.cat(self.test_step_preds)

        auroc = self.test_auroc(pos_score, y.long())
        auprc = self.test_auprc(pos_score, y.long())

        self.log("test_auroc", auroc, on_epoch=True, logger=True)
        self.log("test_auprc", auprc, on_epoch=True, logger=True)

        self.test_step_label.clear()
        self.test_step_preds.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=self.wd)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer=optimizer,
            eta_min=0,
            T_max=self.max_epochs,
        )
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [176]:
from types import SimpleNamespace

config = SimpleNamespace(
    bert_model_name="google/bert_uncased_L-2_H-128_A-2",
    pred_embed_dim=128,      # must match BERT hidden size for tiny
    pred_hidden_dim=128,     # GRU hidden dim
    max_event_len=511,       # max #events per stay
    rnn_layer=1,
    init_bert_random=False,  # use pretrained
    task="binary",           # 1-logit output
)

In [177]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

# paths you already have
dataset_path = "./desc_gen_dataset/"
data_idx_path = "../downstream_idx.parquet"

# datasets
train_ds = DescGenDataset(
    dataset_path=dataset_path,
    data_idx_path=data_idx_path,
    task="y_mort",                # or your label column
    main_window="within48_descemb",
    split="train",
    max_word_len=32,
    max_events=511,
)

val_ds = DescGenDataset(
    dataset_path=dataset_path,
    data_idx_path=data_idx_path,
    task="y_mort",
    main_window="within48_descemb",
    split="val",
    max_word_len=32,
    max_events=511,
)

# collator
tokenizer = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")
collator = DescGenCollator(pad_token_id=tokenizer.pad_token_id)

# dataloaders
train_dl = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    collate_fn=collator,
)

val_dl = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    collate_fn=collator,
)

Loading dataset from disk:   0%|          | 0/24 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/24 [00:00<?, ?it/s]

In [178]:
# batch = next(iter(train_dl))
# for k, v in batch.items():
#     print(k, v.shape if hasattr(v, "shape") else type(v))

model = DescGenEvalModel(
    config=config,
    lr=2e-5,
    wd=0.01,
    max_epochs=10,
    dropout=0.1,
    freeze=False,   # BERT-FT (full fine-tuning)
)

# logits = model(
#     input_ids=batch["input_ids"],
#     attention_mask=batch["attention_mask"],
#     seq_len=batch["seq_len"],
# )
# print("logits shape:", logits)  # should be (B,)

In [179]:
trainer = lt.Trainer(
    max_epochs=1,                              # one full pass over train_dl
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    log_every_n_steps=50,                      # adjust as you like
)

trainer.fit(
    model=model,
    train_dataloaders=train_dl,
    val_dataloaders=val_dl,
)

/home/sas10092/.conda/envs/med-ehr/lib/python3.9/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/sas10092/.conda/envs/med-ehr/lib/python3.9/sit ...
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name        | Type                   | Params | Mode 
---------------------------------------------------------------
0 | encoder     | BertEventEncoder       | 4.4 M  | train
1 | classifier  | GRUEventHead           | 99.2 K | train
2 | criterion   | BCEWithLogitsLoss      | 0      | train
3 | train_auroc | BinaryAUROC            | 0      | train
4 | train_auprc | BinaryA

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

# GenHPF

In [180]:
import torch
from torch.utils.data import Dataset
from typing import Any, Dict, List, Optional, Tuple

import polars as pl
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer


class HierarchicalGenHPFDataset(Dataset):
    def __init__(
        self,
        hf_dataset: HFDataset,
        data_idx_path: str,
        seq_field: str,
        label_field: Optional[str] = None,
        split: str = "train",
        tokenizer_name: str = "emilyalsentzer/Bio_ClinicalBERT",
        max_events: int = 256,
        max_tokens: int = 128,
    ) -> None:
        self.hf_dataset = hf_dataset
        self.seq_field = seq_field
        self.label_field = label_field
        self.max_events = max_events
        self.max_tokens = max_tokens

        df = pl.scan_parquet(data_idx_path).collect()
        self.data_idx = df.filter(pl.col("split") == split).to_pandas()

        subj = self.hf_dataset["subject_id"]
        stay = self.hf_dataset["icustay_id"]
        key_to_hf_idx: Dict[Tuple[int, int], int] = {}
        for i, (s, h) in enumerate(zip(subj, stay)):
            key_to_hf_idx[(int(s), int(h))] = i
        self.key_to_hf_idx = key_to_hf_idx


        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    def __len__(self) -> int:
        return len(self.data_idx)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.data_idx.iloc[idx]
        subject_id = int(row["subject_id"])
        icustay_id = int(row["icustay_id"])

        key = (subject_id, icustay_id)
        if key not in self.key_to_hf_idx:
            return {"input_ids": None, "label": None}

        hf_idx = self.key_to_hf_idx[key]
        hf_row = self.hf_dataset[hf_idx]

        events: List[str] = hf_row[self.seq_field]

        if len(events) > self.max_events:
            events = events[: self.max_events]


        if len(events) == 0:
            return {"input_ids": None, "label": None}

        enc = self.tokenizer(
            events,
            padding="max_length",
            truncation=True,
            max_length=self.max_tokens,
            add_special_tokens=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].long() 

        out: Dict[str, Any] = {
            "input_ids": input_ids,
        }

        if self.label_field is not None:
            y = row[self.label_field]
            out["label"] = torch.tensor(float(y), dtype=torch.float32)

        return out

In [181]:
import math
import torch
from torch import nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float, max_len: int):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)                      # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(1, max_len, d_model)                              # (1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)  # not a parameter, but moves with device

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)

In [182]:
from typing import List


class GenHPFEvalCollator:
    def __init__(self, pad_token_id: int = 0) -> None:
        self.pad_token_id = pad_token_id

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        # drop empty samples
        batch = [b for b in batch if b["input_ids"] is not None]
        if len(batch) == 0:
            return {}

        input_ids_list = [b["input_ids"] for b in batch]  
        sizes = [x.size(0) for x in input_ids_list]
        B = len(input_ids_list)
        S_max = max(sizes)
        W = input_ids_list[0].size(1)

        # (B, S_max, W)
        collated_input_ids = input_ids_list[0].new_full(
            (B, S_max, W), fill_value=self.pad_token_id
        ).long()

        # True = padded event
        padding_mask = torch.ones(B, S_max, dtype=torch.bool)

        for i, (ids, S_i) in enumerate(zip(input_ids_list, sizes)):
            collated_input_ids[i, :S_i, :] = ids
            padding_mask[i, :S_i] = False

        out: Dict[str, Any] = {
            "input_ids": collated_input_ids,
            "padding_mask": padding_mask,
        }

        if "label" in batch[0] and batch[0]["label"] is not None:
            labels = torch.stack([b["label"] for b in batch])  # (B,)
            out["label"] = labels

        return out

In [183]:
import torch
from typing import List, Dict, Any


class GenHPFSimCLRCollator:
    def __init__(
        self,
        pad_token_id: int,
        mask_token_id: int,
        mask_prob: float = 0.15,
    ) -> None:
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.mask_prob = mask_prob

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = [b for b in batch if b["input_ids"] is not None]
        if len(batch) == 0:
            return {}

        views: List[torch.Tensor] = []

        for b in batch:
            ev = b["input_ids"]  # (S, W)
            S = ev.size(0)
            if S <= 1:
                
                views.append(ev)
                views.append(ev)
            else:
                mid = S // 2
                v1 = ev[:mid, :]  
                v2 = ev[mid:, :]  
                views.append(v1)
                views.append(v2)

        sizes = [v.size(0) for v in views]
        B2 = len(views)
        S_max = max(sizes)
        W = views[0].size(1)

        collated_input_ids = views[0].new_full(
            (B2, S_max, W), fill_value=self.pad_token_id
        ).long()
        padding_mask = torch.ones(B2, S_max, dtype=torch.bool)  

        for i, (v, S_i) in enumerate(zip(views, sizes)):
            collated_input_ids[i, :S_i, :] = v
            padding_mask[i, :S_i] = False

        ids = collated_input_ids
        rand = torch.rand_like(ids, dtype=torch.float32)
        mask = rand < self.mask_prob
        mask &= ids != self.pad_token_id
        ids[mask] = self.mask_token_id

        return {
            "input_ids": ids,        
            "padding_mask": padding_mask,  
        }

In [184]:
from datasets import load_from_disk

hf_ds = dx

train_dataset = HierarchicalGenHPFDataset(
    hf_dataset=dx,
    data_idx_path="../downstream_idx.parquet",
    seq_field="within48_genhpf",
    label_field='y_mort',  # unsupervised
    split="train",
    tokenizer_name="emilyalsentzer/Bio_ClinicalBERT",
    max_events=511,
    max_tokens=128,
)

# simclr_collator = GenHPFSimCLRCollator(pad_token_id=train_dataset.tokenizer.pad_token_id,
#                                        mask_token_id=train_dataset.tokenizer.mask_token_id)


eval_collator = GenHPFEvalCollator(pad_token_id=0)
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=eval_collator,
)

In [185]:
from typing import Tuple, Optional


class GenHPFEncoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        pad_token_id: int,
        encoder_embed_dim: int = 128,
        encoder_layers: int = 2,
        encoder_ffn_embed_dim: int = 512,
        encoder_attention_heads: int = 4,
        agg_embed_dim: int = 128,
        agg_layers: int = 4,
        agg_ffn_embed_dim: int = 512,
        agg_attention_heads: int = 4,
        dropout: float = 0.1,
        max_token_len: int = 128,
        max_events: int = 256,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id
        self.encoder_embed_dim = encoder_embed_dim
        self.agg_embed_dim = agg_embed_dim

        self.word_embeddings = nn.Embedding(
            vocab_size, encoder_embed_dim, padding_idx=pad_token_id
        )

        
        self.token_pos_encoding = PositionalEncoding(
            d_model=encoder_embed_dim, dropout=dropout, max_len=max_token_len
        )

        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=encoder_embed_dim,
            nhead=encoder_attention_heads,
            dim_feedforward=encoder_ffn_embed_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.event_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=encoder_layers
        )

        
        self.post_encode_proj = nn.Linear(encoder_embed_dim, agg_embed_dim)

        
        self.event_pos_encoding = PositionalEncoding(
            d_model=agg_embed_dim, dropout=dropout, max_len=max_events
        )

        agg_layer = nn.TransformerEncoderLayer(
            d_model=agg_embed_dim,
            nhead=agg_attention_heads,
            dim_feedforward=agg_ffn_embed_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.event_aggregator = nn.TransformerEncoder(
            agg_layer, num_layers=agg_layers
        )

        self.event_layer_norm = nn.LayerNorm(encoder_embed_dim, eps=1e-12)
        self.agg_layer_norm = nn.LayerNorm(agg_embed_dim, eps=1e-12)

    def forward(
        self,
        input_ids: torch.Tensor,         
        padding_mask: Optional[torch.Tensor] = None,  
        encoder_only: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, S, W = input_ids.shape

        flat_ids = input_ids.view(B * S, W)


        x_tok = self.word_embeddings(flat_ids)
        x_tok = self.token_pos_encoding(x_tok)
        x_tok = self.event_layer_norm(x_tok)

        token_pad_mask = flat_ids.eq(self.pad_token_id) 


        x_tok = self.event_encoder(
            x_tok, src_key_padding_mask=token_pad_mask
        ) 

        if token_pad_mask.any():
            x_tok = x_tok.masked_fill(token_pad_mask.unsqueeze(-1), 0.0)
            lengths = (~token_pad_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)
        else:
            lengths = torch.full(
                (B * S, 1), W, device=x_tok.device, dtype=torch.long
            )

        event_emb = x_tok.sum(dim=1) / lengths 

    
        event_emb = self.post_encode_proj(event_emb) 
        event_emb = event_emb.view(B, S, -1)        

        if padding_mask is None:
            padding_mask = input_ids.eq(self.pad_token_id).all(dim=2)  


        event_emb = self.event_pos_encoding(event_emb)
        event_emb = self.agg_layer_norm(event_emb)

        if encoder_only:
            return event_emb, padding_mask

        x = self.event_aggregator(
            event_emb, src_key_padding_mask=padding_mask
        ) 

        return x, padding_mask

In [186]:
class GenHPFSimCLRModel(nn.Module):
    def __init__(self, encoder: GenHPFEncoder):
        super().__init__()
        self.encoder = encoder

    def forward(
        self,
        input_ids: torch.Tensor,       
        padding_mask: torch.Tensor,    
    ) -> torch.Tensor:
        x, pad_mask = self.encoder(input_ids, padding_mask=padding_mask, encoder_only=False)

        if pad_mask is not None and pad_mask.any():
            x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        lengths = (~pad_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)  
        pooled = x.sum(dim=1) / lengths                              

        return pooled  

In [187]:
class GenHPFClassifier(nn.Module):

    def __init__(
        self,
        encoder: GenHPFEncoder,
        num_outputs: int = 1,   # 1 for binary, >1 for multi-class
    ):
        super().__init__()
        self.encoder = encoder
        self.num_outputs = num_outputs

        self.classifier = nn.Linear(encoder.agg_embed_dim, num_outputs)

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ) -> torch.Tensor:

        x, pad_mask = self.encoder(input_ids, padding_mask=padding_mask, encoder_only=False)


        if pad_mask is not None and pad_mask.any():
            x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)


        lengths = (~pad_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)
        pooled = x.sum(dim=1) / lengths

        logits = self.classifier(pooled)  
        if self.num_outputs == 1:
            logits = logits.squeeze(-1) 
        return logits

In [188]:
import pytorch_lightning as lt
import torch
from torch import nn
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision

In [189]:
class GenHPFDownstreamModule(lt.LightningModule):
    def __init__(
        self,
        encoder: GenHPFEncoder,
        num_outputs: int = 1,
        lr: float = 2e-5,
        wd: float = 1e-3,
        max_epochs: int = 100,
        pos_weight: float = 1.0,  
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["encoder"])

        self.model = GenHPFClassifier(
            encoder=encoder,
            num_outputs=num_outputs,
        )

        # loss
        if num_outputs == 1:
            self.criterion = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor(pos_weight)
            )
        else:
            self.criterion = nn.CrossEntropyLoss()

        if num_outputs == 1:
            self.train_auroc = BinaryAUROC()
            self.train_auprc = BinaryAveragePrecision()
            self.val_auroc = BinaryAUROC()
            self.val_auprc = BinaryAveragePrecision()
            self.test_auroc = BinaryAUROC()
            self.test_auprc = BinaryAveragePrecision()

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs

        self.train_step_preds = []
        self.train_step_label = []
        self.val_step_preds = []
        self.val_step_label = []
        self.test_step_preds = []
        self.test_step_label = []

    def forward(self, input_ids, padding_mask):
        logits = self.model(input_ids=input_ids, padding_mask=padding_mask)
        return logits


    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]       
        padding_mask = batch["padding_mask"] 
        y = batch["label"].float().view(-1) 

        logits = self.forward(input_ids, padding_mask) 

        if self.hparams.num_outputs == 1:
            loss = self.criterion(logits, y)
            pos_score = torch.sigmoid(logits)
            self.train_step_label.append(y.detach())
            self.train_step_preds.append(pos_score.detach())
        else:
            y_long = y.long()
            loss = self.criterion(logits, y_long)

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_train_epoch_end(self):
        if self.hparams.num_outputs != 1:
            return

        y = torch.cat(self.train_step_label)
        pos_score = torch.cat(self.train_step_preds)

        auroc = self.train_auroc(pos_score, y.long())
        auprc = self.train_auprc(pos_score, y.long())

        self.log("train_auroc", auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log("train_auprc", auprc, on_epoch=True, logger=True, prog_bar=True)

        self.train_step_label.clear()
        self.train_step_preds.clear()

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        padding_mask = batch["padding_mask"]
        y = batch["label"].float().view(-1)

        logits = self.forward(input_ids, padding_mask)

        if self.hparams.num_outputs == 1:
            loss = self.criterion(logits, y)
            pos_score = torch.sigmoid(logits)
            self.val_step_label.append(y.detach())
            self.val_step_preds.append(pos_score.detach())
        else:
            y_long = y.long()
            loss = self.criterion(logits, y_long)

        self.log("val_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_validation_epoch_end(self):
        if self.hparams.num_outputs != 1:
            return

        y = torch.cat(self.val_step_label)
        pos_score = torch.cat(self.val_step_preds)

        auroc = self.val_auroc(pos_score, y.long())
        auprc = self.val_auprc(pos_score, y.long())

        self.log("val_auroc", auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log("val_auprc", auprc, on_epoch=True, logger=True, prog_bar=True)

        self.val_step_label.clear()
        self.val_step_preds.clear()


    def test_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        padding_mask = batch["padding_mask"]
        y = batch["label"].float().view(-1)

        logits = self.forward(input_ids, padding_mask)

        if self.hparams.num_outputs == 1:
            loss = self.criterion(logits, y)
            pos_score = torch.sigmoid(logits)
            self.test_step_label.append(y.detach())
            self.test_step_preds.append(pos_score.detach())
        else:
            y_long = y.long()
            loss = self.criterion(logits, y_long)

        self.log("test_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_test_epoch_end(self):
        if self.hparams.num_outputs != 1:
            return

        y = torch.cat(self.test_step_label)
        pos_score = torch.cat(self.test_step_preds)

        auroc = self.test_auroc(pos_score, y.long())
        auprc = self.test_auprc(pos_score, y.long())

        self.log("test_auroc", auroc, on_epoch=True, logger=True)
        self.log("test_auprc", auprc, on_epoch=True, logger=True)

        self.test_step_label.clear()
        self.test_step_preds.clear()

    def configure_optimizers(self):
        optimizer = SGD(self.parameters(), lr=self.lr, weight_decay=self.wd)
        scheduler = CosineAnnealingLR(
            optimizer=optimizer,
            eta_min=0.0,
            T_max=self.max_epochs,
        )
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [190]:
import torch.nn.functional as F


class GenHPFSimCLRModule(lt.LightningModule):
    def __init__(
        self,
        encoder: GenHPFEncoder,
        lr: float = 1e-3,
        wd: float = 1e-4,
        max_epochs: int = 100,
        temperature: float = 0.5,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["encoder"])

        self.model = GenHPFSimCLRModel(encoder=encoder)
        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs
        self.temperature = temperature

    def forward(self, input_ids, padding_mask):
        # returns embeddings (2B, D)
        return self.model(input_ids=input_ids, padding_mask=padding_mask)

    def _nt_xent_loss(self, z: torch.Tensor) -> torch.Tensor:

        B2, D = z.shape
        assert B2 % 2 == 0, ""
        B = B2 // 2

        z1 = z[:B]
        z2 = z[B:]


        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)


        representations = torch.cat([z1, z2], dim=0)  


        sim = torch.matmul(representations, representations.T) 
        sim = sim / self.temperature


        diag_mask = torch.eye(2 * B, device=sim.device, dtype=torch.bool)
        sim.masked_fill_(diag_mask, float("-inf"))

        labels = torch.arange(2 * B, device=sim.device)
        labels = (labels + B) % (2 * B)  

        
        loss = F.cross_entropy(sim, labels)

        return loss

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]      
        padding_mask = batch["padding_mask"]

        z = self.forward(input_ids, padding_mask)  
        loss = self._nt_xent_loss(z)

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def configure_optimizers(self):
        optimizer = SGD(self.parameters(), lr=self.lr, weight_decay=self.wd)
        scheduler = CosineAnnealingLR(
            optimizer=optimizer,
            eta_min=0.0,
            T_max=self.max_epochs,
        )
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [191]:
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as lt
from datasets import load_from_disk
from transformers import AutoTokenizer

# ------------- tokenizer -------------
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

# ------------- encoder -------------
encoder = GenHPFEncoder(
    vocab_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    encoder_embed_dim=128,
    encoder_layers=2,
    encoder_ffn_embed_dim=512,
    encoder_attention_heads=4,
    agg_embed_dim=128,
    agg_layers=4,
    agg_ffn_embed_dim=512,
    agg_attention_heads=4,
    dropout=0.1,
    max_token_len=128,   # must match your max_tokens
    max_events=256,      # must match your max_events
)

# ------------- HF dataset & index -------------
hf_ds = load_from_disk("./desc_gen_dataset/")          # your HF dataset
data_idx_path = "../downstream_idx.parquet"               # parquet with labels + split

train_dataset_genhpf = HierarchicalGenHPFDataset(
    hf_dataset=hf_ds,
    data_idx_path=data_idx_path,
    seq_field="within48_genhpf",  
    label_field="y_mort",         
    split="train",
    tokenizer_name="emilyalsentzer/Bio_ClinicalBERT",
    max_events=256,
    max_tokens=128,
)

val_dataset_genhpf = HierarchicalGenHPFDataset(
    hf_dataset=hf_ds,
    data_idx_path=data_idx_path,
    seq_field="within48_genhpf",
    label_field="y_mort",
    split="valid",
    tokenizer_name="emilyalsentzer/Bio_ClinicalBERT",
    max_events=256,
    max_tokens=128,
)

# For SimCLR (unsupervised) you can reuse same dataset but ignore labels (label_field=None)
simclr_dataset_genhpf = HierarchicalGenHPFDataset(
    hf_dataset=hf_ds,
    data_idx_path=data_idx_path,
    seq_field="within48_genhpf",
    label_field=None,
    split="train",
    tokenizer_name="emilyalsentzer/Bio_ClinicalBERT",
    max_events=256,
    max_tokens=128,
)

Loading dataset from disk:   0%|          | 0/24 [00:00<?, ?it/s]

In [193]:
# ------------- SimCLR collator & loader -------------
simclr_collator = GenHPFSimCLRCollator(
    pad_token_id=tokenizer.pad_token_id,
    mask_token_id=tokenizer.mask_token_id,
    mask_prob=0.15,
)

simclr_loader = DataLoader(
    simclr_dataset_genhpf,
    batch_size=8,          # small for sanity check
    shuffle=True,
    num_workers=4,
    collate_fn=simclr_collator,
)

# ------------- SimCLR Lightning module -------------
simclr_module = GenHPFSimCLRModule(
    encoder=encoder,
    lr=1e-3,
    wd=1e-4,
    max_epochs=2,
    temperature=0.5,
)

# ------------- Trainer (small run) -------------
trainer_simclr = lt.Trainer(
    max_epochs=1,
    limit_train_batches=5,   # only 5 batches to smoke-test
    accelerator="auto",
    devices=1,
    log_every_n_steps=1,
)

trainer_simclr.fit(simclr_module, train_dataloaders=simclr_loader)

/home/sas10092/.conda/envs/med-ehr/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/sas10092/.conda/envs/med-ehr/lib/python3.9/sit ...
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/sas10092/.conda/envs/med-ehr/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the defaul

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


In [195]:
# ------------- eval collator & loaders -------------
eval_collator = GenHPFEvalCollator(pad_token_id=tokenizer.pad_token_id)

train_loader = DataLoader(
    train_dataset_genhpf,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    collate_fn=eval_collator,
)

val_loader = DataLoader(
    val_dataset_genhpf,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    collate_fn=eval_collator,
)


downstream_module = GenHPFDownstreamModule(
    encoder=encoder,      
    num_outputs=1,         
    lr=2e-4,
    wd=1e-3,
    max_epochs=5,
    pos_weight=1.0,        
)


trainer_downstream = lt.Trainer(
    max_epochs=1,
    limit_train_batches=5,
    limit_val_batches=2,
    accelerator="auto",
    devices=1,
    log_every_n_steps=1,
)

trainer_downstream.fit(
    downstream_module,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name        | Type                   | Params | Mode 
---------------------------------------------------------------
0 | model       | GenHPFClassifier       | 4.9 M  | train
1 | criterion   | BCEWithLogitsLoss      | 0      | train
2 | train_auroc | BinaryAUROC            | 0      | train
3 | train_auprc | BinaryAveragePrecision | 0      | train
4 | val_auroc   | BinaryAUROC            | 0      | train
5 | val_auprc   | BinaryAveragePrecision | 0      | train
6 | test_auroc  | BinaryAUROC            | 0      | train
7 | test_auprc  | BinaryAveragePrecision | 0      | train
---------------------------------------------------------------
4.9 M     Trainable params
0         Non-trainable params
4.9 M     Total params

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/sas10092/.conda/envs/med-ehr/lib/python3.9/site-packages/pytorch_lightning/utilities/data.py:106: Total length of `DataLoader` across ranks is zero. Please make sure this was your intention.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disabl

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.
